# Resume Screening Automation

**Context:** With 388 applicants and a 4-day window to produce an interview shortlist for a data science / ML engineering role, this pipeline was designed and built from scratch to turn an unstructured folder of PDFs into a ranked, reviewable scorecard.

**Pipeline overview:**
1. Bulk PDF ingestion via `pdfplumber`
2. Text normalization + optional manual overrides
3. Regex-based feature extraction (degree, YOE, GCP cert, 20+ JD-specific signals)
4. Weighted composite scoring calibrated to the job description
5. CSV export + score distribution visualization

> **Note:** Resume PDFs and output CSVs are not included in this repository (candidate privacy). Update `RESUME_FOLDER`, `MASTER_CSV`, `OUTPUT_CSV`, and optionally `OVERRIDES_CSV` in the **Configuration** cell below before running.

In [ ]:
# ============================================================
# Configuration — update these paths before running
# ============================================================
import os

# Folder containing resume PDFs
RESUME_FOLDER = os.path.join(os.path.expanduser("~"), "data", "resumes")

# Output directory
RESULTS_DIR = os.path.join(os.path.expanduser("~"), "data", "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

MASTER_CSV    = os.path.join(RESULTS_DIR, "master.csv")
OUTPUT_CSV    = os.path.join(RESULTS_DIR, "analyzed_resumes_final.csv")
OVERRIDES_CSV = os.path.join(RESULTS_DIR, "manual_resume_overrides.csv")  # optional


In [ ]:
import pandas as pd
import csv
import os
import re
from datetime import datetime

import matplotlib.pyplot as plt

try:
    import pdfplumber
except ModuleNotFoundError:
    print("pdfplumber not found, installing...")
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pdfplumber"])
    import pdfplumber

print('Imports loaded.')


In [ ]:
# Ingest all PDFs from RESUME_FOLDER — one row per page
resumes_text = []
file_number = 1

for fn in os.listdir(RESUME_FOLDER):
    filepath = os.path.join(RESUME_FOLDER, fn)

    if not os.path.isfile(filepath):
        continue

    if not fn.lower().endswith(".pdf"):
        continue

    with pdfplumber.open(filepath) as pdf:
        for page in pdf.pages:
            page_to_text = page.extract_text() or ""
            resumes_text.append({
                "filename": fn,
                "resume_text": page_to_text
            })

    print(f"PDF # {file_number}")
    file_number += 1

print(f"\nTotal PDFs ingested: {file_number - 1}")


In [ ]:
# Save raw extracted text to master CSV before analysis
pd.DataFrame(resumes_text).to_csv(
    MASTER_CSV,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_ALL,
    escapechar="\\",
    lineterminator="\n"
)
print(f"Saved: {MASTER_CSV}")


In [ ]:
# ============================================================
# Resume analysis — scoring pipeline
# ============================================================

# ---------- Load safely (multiline + quotes) ----------
df = pd.read_csv(
    MASTER_CSV,
    dtype={"filename": "string", "resume_text": "string"},
    keep_default_na=False,
    encoding="utf-8",
    engine="python",
    quoting=csv.QUOTE_MINIMAL
)

# ---------- Combine multi-page resumes ----------
combined_df = (
    df.groupby("filename", as_index=False)["resume_text"]
      .apply(lambda s: "\n".join([t for t in s.astype(str).tolist() if t.strip()]))
)

# ==================================================
# Manual overrides (optional)
# If OVERRIDES_CSV exists, its resume_text values replace extracted text for matching filenames.
# Useful when PDF extraction fails for specific files.
# ==================================================
try:
    overrides_df = pd.read_csv(
        OVERRIDES_CSV,
        dtype={"filename": "string", "resume_text": "string"},
        keep_default_na=False,
        encoding="utf-8"
    )
    overrides_map = dict(zip(overrides_df["filename"], overrides_df["resume_text"]))
    combined_df["resume_text"] = combined_df.apply(
        lambda r: overrides_map.get(r["filename"], r["resume_text"]),
        axis=1
    )
    combined_df["manual_override"] = combined_df["filename"].isin(overrides_map)
    print(f"Applied {len(overrides_map)} manual override(s).")
except FileNotFoundError:
    combined_df["manual_override"] = 0
    print("No overrides file found — proceeding without manual overrides.")

# ---------- Text normalization ----------
def normalize(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

# ---------- Degree patterns ----------
DEGREE_PATTERNS = {
    "masters": re.compile(
        r"""
        (?<![a-z])
        (
            m\s*\.?\s*s\s*\.?
            |m\s*\.?\s*sc\s*\.?
            |m\s*\.?\s*eng\s*\.?
            |m\s*\.?\s*a\s*\.?
            |mba
            |master'?s?
        )
        (?![a-z])
        """,
        re.I | re.VERBOSE
    ),
    "bachelors": re.compile(
        r"""
        (?<![a-z])
        (
            b\s*\.?\s*s\s*\.?
            |b\s*\.?\s*a\s*\.?
            |bachelor'?s?
            |undergrad|undergraduate
        )
        (?![a-z])
        """,
        re.I | re.VERBOSE
    ),
}
MASTERS_FALSE_POS = re.compile(r"\b(ms office|ms excel|ms word|ms teams)\b", re.I)

SPECIALIZED_FIELDS = re.compile(
    r"\b(artificial intelligence|data science|machine learning|deep learning"
    r"|nlp|computer vision|ai\b|ml\b|analytics)\b",
    re.I
)

GCP_CERT = re.compile(
    r"\b("
    r"google cloud certified|"
    r"professional (data engineer|machine learning engineer|cloud architect"
    r"|cloud developer|devops engineer|security engineer|network engineer)|"
    r"associate cloud engineer|"
    r"cloud digital leader"
    r")\b",
    re.I
)

# ---------- Years of experience ----------
YOE_CONTEXT = re.compile(
    r"\b(\d{1,2})\s*\+?\s*(?:years?|yrs?)\b"
    r"(?:\s+of)?\s+(?:relevant\s+)?(?:professional\s+)?(?:work\s+)?experience\b",
    re.I
)
YOE_CONTEXT_ALT = re.compile(
    r"\b(\d{1,2})\s*\+?\s*(?:years?|yrs?)\b"
    r"(?:\s*['']?\s*)?(?:\s+of)?\s+experience\b",
    re.I
)
YOE_NEGATIVE_CONTEXT = re.compile(
    r"\b("
    r"life\s+stage|juvenile|adult|senescent|"
    r"aged|age\s+\d|years?\s+old|yrs?\.\)|"
    r"time\s+series|"
    r"historical|history\s+of|"
    r"climate\s+data|"
    r"daily\s+averages?|"
    r"forecast(ed|ing)?|"
    r"modeled|modeling|"
    r"dataset|data\s+set|"
    r"longitudinal|"
    r"\d+\s*-\s*year"
    r")\b",
    re.I
)

def extract_yoe_total(text: str) -> int:
    t = text.lower()
    years = []
    for m in YOE_CONTEXT.finditer(t):
        years.append(int(m.group(1)))
    for m in YOE_CONTEXT_ALT.finditer(t):
        years.append(int(m.group(1)))
    years = [y for y in years if 0 < y < 45]
    if years:
        return max(years)
    generic = list(re.finditer(r"\b(\d{1,2})\s*\+?\s*(?:years?|yrs?)\b", t, flags=re.I))
    for m in generic:
        y = int(m.group(1))
        if not (0 < y < 45):
            continue
        window = t[max(0, m.start() - 40): min(len(t), m.end() + 40)]
        if YOE_NEGATIVE_CONTEXT.search(window):
            continue
        years.append(y)
    years = [y for y in years if 0 < y < 45]
    return max(years) if years else 0

# ---------- Degree + cert ----------
def extract_degree(text: str) -> str:
    t = text.lower()
    phd_pattern = r"\b(ph\.?d\.?|doctorate|doctoral|d\.phil)\b"
    if re.search(phd_pattern, t):
        return "PhD / Doctorate"
    m = DEGREE_PATTERNS["masters"].search(t)
    if m and not MASTERS_FALSE_POS.search(t):
        start = max(0, m.start() - 40)
        end = min(len(t), m.end() + 140)
        window = t[start:end]
        return "Master's (AI / DS / ML)" if SPECIALIZED_FIELDS.search(window) else "Master's (Other)"
    if DEGREE_PATTERNS["bachelors"].search(t):
        return "Bachelor's"
    return "None"

def extract_cert(text: str) -> str:
    m = GCP_CERT.search(text)
    return m.group(0) if m else ""

# ---------- Signals ----------
MENTION_CAP = 10  # cap per-signal mention count to prevent verbosity inflation

SKILLS = {
    "sql": re.compile(r"\b(sql|postgres|mysql|sql server|t-?sql|bigquery|snowflake|redshift)\b", re.I),
    "ml_ai": re.compile(
        r"\b("
        r"machine learning|deep learning|artificial intelligence|"
        r"modeling|classification|regression|nlp|computer vision|"
        r"tensorflow|pytorch|sklearn|xgboost"
        r")\b",
        re.I
    ),
    "gcp": re.compile(
        r"\b("
        r"gcp|google cloud|"
        r"vertex ai|bigquery|"
        r"cloud run|cloud functions|"
        r"dataflow|composer"
        r")\b",
        re.I
    ),
    "python": re.compile(r"\bpython\b", re.I),
}

# JD-specific signals — tuned after reviewing what applicants actually write
JD_SIGNALS = {
    "vertex_ai":          re.compile(r"\b(vertex ai|vertexai)\b", re.I),
    "gemini_enterprise":  re.compile(r"\b(gemini enterprise)\b", re.I),
    "google_adk":         re.compile(r"\b(google adk|agent development kit)\b", re.I),
    "agentic_ai":         re.compile(r"\b(agentic ai|ai agents?|autonomous agents?)\b", re.I),
    "mlops":              re.compile(r"\b(mlops|model monitoring|model registry|pipeline|pipelines|orchestration|ci/cd)\b", re.I),
    "drift_monitoring":   re.compile(r"\b(data drift|concept drift|model drift|drift detection)\b", re.I),
    "deployment":         re.compile(r"\b(deploy(ed|ment)?|serving|endpoint|inference api|rest api|model endpoint)\b", re.I),
    "feature_engineering":re.compile(r"\b(feature engineering|feature store)\b", re.I),
    "data_modeling":      re.compile(r"\b(data model(ing)?|schema|erd|dimensional model|star schema)\b", re.I),
    "documentation":      re.compile(r"\b(documentation|runbook|sop|confluence|wiki)\b", re.I),
    "user_training":      re.compile(r"\b(training|workshop|enablement)\b", re.I),
    "database_admin":     re.compile(r"\b(database administration|dba|indexing|query optimization)\b", re.I),
    "data_definition":    re.compile(r"\b(data definition|ddl|schema design)\b", re.I),
    "data_conversion":    re.compile(r"\b(data conversion|etl|elt|data migration)\b", re.I),
    "unstructured_data":  re.compile(r"\b(unstructured|semi-structured|json|xml|text data)\b", re.I),
}

ALL_SIGNALS = {**SKILLS, **JD_SIGNALS}

def extract_signals(text: str) -> dict:
    t = text.lower()
    out = {}
    for key, pattern in ALL_SIGNALS.items():
        matches = pattern.findall(t)
        out[f"has_{key}"] = int(len(matches) > 0)
        out[f"{key}_mentions"] = min(len(matches), MENTION_CAP)
    return out

def analyze_resume(text: str, manual_override: int) -> pd.Series:
    text = normalize(text)
    degree = extract_degree(text)
    yoe_total = extract_yoe_total(text)
    gcp_cert = extract_cert(text)
    signals = extract_signals(text)
    warnings = []
    if manual_override:
        warnings.append("MANUAL_TEXT_OVERRIDE")
    if len(text) < 300:
        warnings.append("LOW_TEXT_EXTRACT")
    if "\x00" in text:
        warnings.append("NULL_BYTES")
    warn = ";".join(warnings)
    return pd.Series({
        "Degree": degree,
        "yoe_total": yoe_total,
        "gcp_cert": gcp_cert,
        **signals,
        "warnings": warn
    })

# ---------- Apply ----------
features_df = combined_df.apply(
    lambda r: analyze_resume(r["resume_text"], r["manual_override"]),
    axis=1
)
final_output = pd.concat([combined_df[["filename"]], features_df], axis=1)

# ---------- Score ----------
DEGREE_SCORE = {
    "None":                   0,
    "Bachelor's":             0,
    "Master's (Other)":       1,
    "Master's (AI / DS / ML)": 2,
    "PhD / Doctorate":        3,
}
final_output["degree_score"] = final_output["Degree"].map(DEGREE_SCORE).fillna(0)
final_output["has_gcp_cert"] = (final_output["gcp_cert"] != "").astype(int)

final_output["tech_signal_score"] = (
    final_output["has_sql"]              * 2 +
    final_output["has_ml_ai"]            * 3 +
    final_output["has_gcp"]              * 2 +
    final_output["has_gcp_cert"]         * 3 +
    # JD-specific signals
    final_output["has_vertex_ai"]        * 3 +
    final_output["has_agentic_ai"]       * 3 +
    final_output["has_google_adk"]       * 4 +
    final_output["has_gemini_enterprise"] * 4 +
    final_output["has_mlops"]            * 2 +
    final_output["has_deployment"]       * 2 +
    final_output["has_drift_monitoring"] * 1 +
    # Depth: capped at 5 to reward presence, not verbosity
    final_output["ml_ai_mentions"].clip(upper=5) +
    # Education (preferred, not dominant)
    final_output["degree_score"]
)

# ---------- Reorder + drop mention columns ----------
base_cols = ["filename", "Degree", "degree_score", "has_gcp_cert", "gcp_cert"]
other_cols = [c for c in final_output.columns if c not in base_cols]
final_output = final_output[base_cols + other_cols]
cols_to_drop = [c for c in final_output.columns if c.endswith("_mentions")]
final_output = final_output.drop(columns=cols_to_drop)

# ---------- Save ----------
final_output.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_ALL,
    escapechar="\\",
    lineterminator="\n"
)
print(f"\nSaved: {OUTPUT_CSV}")
print(f"Total candidates scored: {len(final_output)}")


In [ ]:
# Score distribution
df_viz = final_output.copy()

bins = [0, 12, 24, 36]
labels = ["0 to 12", "13 to 24", "25 to 36"]

df_viz["score_bin"] = pd.cut(
    df_viz["tech_signal_score"],
    bins=bins,
    include_lowest=True,
    right=True,
    labels=labels
)

df_viz["score_bin"] = df_viz["score_bin"].astype("category")
df_viz["score_bin"].value_counts().reindex(labels, fill_value=0)


In [ ]:
# GCP presence by score bin
col_name = 'has_gcp'

ct = pd.crosstab(df_viz["score_bin"], df_viz[col_name])
ct_norm = ct.div(ct.sum(axis=1), axis=0)

ct_norm.plot(kind="barh", stacked=True, figsize=(8, 4))
plt.xlabel("Proportion of candidates")
plt.ylabel("Tech signal score bin")
plt.legend(title=col_name, loc="center left", bbox_to_anchor=(1.02, 0.5), labels=["No", "Yes"])
plt.tight_layout()
plt.show()
